# 03 — Gap Detection
Identify coverage failures: poor RSRP, negative SINR, handover stress zones.
This is the 'real gold' — where the network fails vehicles in transit.

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan')
import pandas as pd
import folium
from pipeline.ingest import load_raw, clean
from pipeline.gaps import detect_poor_signal, detect_handover_stress, gap_summary
from pipeline.handovers import extract_handovers

df = clean(load_raw())
df = detect_poor_signal(df)
summary = gap_summary(df)
print('Gap Summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# Worst performing cells
worst_cells = (
    df[df['is_gap']]
    .groupby('cell_id')
    .agg(gap_events=('is_gap', 'count'), avg_rsrp=('rsrp', 'mean'), min_rsrp=('rsrp', 'min'))
    .round(1)
    .sort_values('gap_events', ascending=False)
    .head(10)
)
print('Worst 10 cells by gap event count:')
worst_cells

In [ ]:
# Interactive map of gap locations
centre = [df['lat'].mean(), df['long'].mean()]
m = folium.Map(location=centre, zoom_start=13, tiles='CartoDB positron')

# Good signal — blue
for _, row in df[~df['is_gap']].iloc[::5].iterrows():
    folium.CircleMarker([row['lat'], row['long']], radius=2,
                        color='#4A90D9', fill=True, opacity=0.3).add_to(m)

# Gaps — red with detail popup
for _, row in df[df['is_gap']].iterrows():
    reasons = []
    if row['rsrp_gap']: reasons.append(f'RSRP={row["rsrp"]}dBm')
    if row['sinr_gap']: reasons.append(f'SINR={row["sinr"]}dB')
    if row['signal_gap']: reasons.append(f'Signal={row["signal"]}/5')
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=5,
        color='#D0021B',
        fill=True,
        popup=f"Cell {row['cell_id']}<br>" + ' | '.join(reasons)
    ).add_to(m)

m

In [ ]:
# Handover stress zones
handovers = extract_handovers(df)
stress = detect_handover_stress(handovers, window_seconds=30, min_handovers=3)
if len(stress):
    print(f'Handover stress events: {len(stress)}')
    stress.head(10)
else:
    print('No handover stress zones detected with current thresholds')

In [ ]:
print(df["is_gap"].sum())
print(df["rsrp_gap"].sum())
print(df["sinr_gap"].sum())
print(df["signal_gap"].sum())

In [ ]:
print(gap_summary(df))

In [ ]:
df[df["rsrp"] == df["rsrp"].min()][["trip", "time", "lat", "long", "cell_id", "rsrp"]]